In [ ]:
session.sql("DROP TABLE IF EXISTS tmp_cases_stream_snapshot").collect()
print("Temp table dropped")

In [ ]:
session.sql("""
CREATE TEMP TABLE tmp_cases_stream_snapshot AS
SELECT *
FROM STREAM_T_CASES
WHERE METADATA$ACTION = 'INSERT'
""").collect()

cases_raw = session.table("tmp_cases_stream_snapshot")
cases_raw = cases_raw.drop("METADATA$ACTION", "METADATA$ISUPDATE", "METADATA$ROW_ID")
print(f"snapshot cases records found")

In [ ]:
ind_columns = ["RESTRICTED_IND", "CASE_ACCESS_IND"]

cases_clean = cases_raw
for c in ind_columns:
    cases_clean = cases_clean.with_column(
        c,
        when(
            (trim(col(c)).is_null()) | (trim(col(c)) == lit("")),
            lit("N")
        ).otherwise(trim(col(c)))
    )

code_columns = ["CASE_TYPE", "CURRENT_CASE_STATUS_CODE", "RESTRICTION_REASON_CODE"]

for c in code_columns:
    cases_clean = cases_clean.with_column(
        c,
        upper(trim(col(c)))
    )

desc_columns = ["CASE_NAME", "CASE_CMNT", "IMPORTANT_OBSERVATIONS", "RESTRICTION_REASON_CMNT"]

for c in desc_columns:
    cases_clean = cases_clean.with_column(
        c,
        when(
            (trim(col(c)).is_null()) | (trim(col(c)) == lit("")),
            lit("NA")
        ).otherwise(trim(col(c)))
    )

user_columns = ["ADD_USER", "MOD_USER"]

for c in user_columns:
    cases_clean = cases_clean.with_column(
        c,
        when(
            (trim(col(c)).is_null()) | (trim(col(c)) == lit("")),
            lit("SYSTEM")
        ).otherwise(trim(col(c)))
    )

trim_only_columns = ["UC_CASE_NAME", "ASSIST_CASE_NUM", "ARCHIVE_FILE_NAME", "SOURCE_FILE_NAME"]

for c in trim_only_columns:
    cases_clean = cases_clean.with_column(
        c,
        trim(col(c))
    )

cases_clean = cases_clean.with_column_renamed("LOAD_TIMESTAMP", "RAW_LOAD_TIMESTAMP")
print("cases clean")

In [ ]:
valid_cases = cases_clean.filter(
    col("CAS_ID").is_not_null()
)

invalid_cases = cases_clean.filter(
    col("CAS_ID").is_null()
).with_column(
    "ERROR_REASON",
    lit("CAS_ID_NULL")
)
print("seperated valid and invalid records")

In [ ]:
checksum_columns = [
    ("CAS_ID", "number"),
    ("CASE_TYPE", "string"),
    ("CASE_NAME", "string"),
    ("UC_CASE_NAME", "string"),
    ("RESTRICTED_IND", "string"),
    ("CASE_ACCESS_IND", "string"),
    ("ASSIST_CASE_NUM", "string"),
    ("CASE_CMNT", "string"),
    ("IMPORTANT_OBSERVATIONS", "string"),
    ("RESTRICTION_REASON_CODE", "string"),
    ("RESTRICTION_REASON_CMNT", "string"),
    ("CURRENT_CASE_STATUS_CODE", "string"),
    ("ARCHIVE_FILE_NAME", "string"),
    ("ADD_USER", "string"),
    ("MOD_USER", "string"),
    ("UNIT_ORGN_ID", "number"),
    ("AREA_ORGN_ID", "number"),
    ("REGION_ORGN_ID", "number"),
    ("PERSON_PERSON_REQUESTS_ID", "number"),
    ("EXPECTED_CLOSE_DATE", "timestamp"),
    ("CURRENT_CASE_STATUS_DATE", "timestamp"),
    ("RESTRICTION_DATE", "timestamp"),
    ("ARCHIVE_DATE", "timestamp"),
]

checksum_exprs = []
for col_name, col_type in checksum_columns:
    if col_type == "string":
        checksum_exprs.append(coalesce(col(col_name), lit("")))
    else:
        checksum_exprs.append(coalesce(col(col_name).cast("string"), lit("")))

cases_final = valid_cases.with_column(
    "CHECKSUM",
    sha2(concat_ws(lit("|"), *checksum_exprs), 256)
)

In [ ]:
cases_final = cases_final.with_column(
    "STAGING_LOADED_TIMESTAMP",
    current_timestamp()
)

cases_final.write.save_as_table(
    table_name=f"{DB}.{STG}.{STG_CASES}",
    mode="truncate"
)

print(f"CASES loaded into {STG}.{STG_CASES}")

In [ ]:
invalid_cases.create_or_replace_temp_view("tmp_invalid_cases")

invalid_count = invalid_cases.count()

if invalid_count > 0:
    session.sql(f"""
    INSERT INTO {DB}.{STG}.{INVALID_RECORDS}
    (
        TABLE_NAME,
        ERROR_REASON,
        SOURCE_FILE_NAME,
        RAW_LOAD_TIMESTAMP,
        RAW_RECORD
    )
    SELECT
        'T_CASES',
        ERROR_REASON,
        SOURCE_FILE_NAME,
        RAW_LOAD_TIMESTAMP,
        OBJECT_CONSTRUCT(*)
    FROM tmp_invalid_cases
    """).collect()
    print(f"Invalid records saved into {STG}.{INVALID_RECORDS}")
else:
    print("No invalid records")

In [ ]:
rows_processed = cases_final.count()
rows_failed = invalid_count

status = 'SUCCESS' if rows_failed == 0 else 'PARTIAL_SUCCESS'

session.call(
    f"{DB}.{AUDIT}.{SP_LOG_AUDIT}",
    session.sql("SELECT UUID_STRING()").collect()[0][0],
    'STG_CASES_LOAD',
    'STAGING',
    datetime.now(),
    datetime.now(),
    rows_processed,
    rows_failed,
    status,
    None
)

session.call(
    f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}",
    status,
    'STG_CASES_LOAD',
    'STAGING',
    f'CASES staging completed. Rows processed: {rows_processed}, Failed rows: {rows_failed}'
)
print("Audit log inserted and email sent")